In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Set catalog and schema
catalog = "automobile_catalog"
schema = "003_gold"

print(f"Working with: {catalog}.{schema}")

In [0]:
cube_operations_detail = spark.sql(f"""
    WITH deduped_technicians AS (
        SELECT *
        FROM {catalog}.{schema}.dim_technician
        QUALIFY ROW_NUMBER() OVER (PARTITION BY technician_id ORDER BY technician_id) = 1
    )
    SELECT 
        o.store_id,
        s.store_name,
        s.city,
        s.state,
        s.store_type,
        s.manager_name,
        o.technician_id,
        t.technician_name,
        DATE_TRUNC('month', o.vehicle_in_datetime) AS period_month,
        YEAR(o.vehicle_in_datetime) AS period_year,
        MONTH(o.vehicle_in_datetime) AS period_month_num,
        o.service_type,
        o.order_status,
        o.days_in_shop,
        o.days_to_work_start,
        o.work_duration_days,
        o.delivery_variance_days,
        o.is_on_time,
        o.vehicle_in_datetime
    FROM {catalog}.{schema}.fact_orders o
    INNER JOIN {catalog}.{schema}.dim_store s ON o.store_id = s.store_id
    LEFT JOIN deduped_technicians t ON o.technician_id = t.technician_id
    WHERE o.vehicle_in_datetime IS NOT NULL
""")

cube_operations_detail.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{schema}.cube_operations_detail")
print(f"✓ Operations Detail Cube created: {catalog}.{schema}.cube_operations_detail")
print(f"Total rows: {cube_operations_detail.count():,}")
fact_count = spark.table(f'{catalog}.{schema}.fact_orders').filter("vehicle_in_datetime IS NOT NULL").count()
print(f"Expected rows from fact_orders: {fact_count:,}")
ratio = cube_operations_detail.count() / fact_count
print(f"Ratio: {ratio:.1f}x {'✓' if ratio == 1.0 else '❌'}")
display(cube_operations_detail.orderBy("vehicle_in_datetime").limit(10))

In [0]:
cube_financial_detail = spark.sql(f"""
    WITH final_estimates AS (
        SELECT *
        FROM {catalog}.{schema}.fact_estimates
        QUALIFY ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY estimate_date DESC, is_initial_estimate ASC) = 1
    )
    SELECT 
        i.order_id,
        i.store_id,
        s.store_name,
        s.city,
        s.state,
        s.store_type,
        s.manager_name,
        DATE_TRUNC('month', i.invoice_date) AS period_month,
        YEAR(i.invoice_date) AS period_year,
        MONTH(i.invoice_date) AS period_month_num,
        i.order_status,
        i.invoice_date,
        i.invoice_amount,
        b.budget_amount,
        e.estimator_id,
        est.estimator_name,
        e.is_initial_estimate,
        e.estimate_amount AS final_estimate_amount,
        e.actual_amount,
        (e.actual_amount - e.estimate_amount) AS estimate_variance
    FROM {catalog}.{schema}.fact_invoices i
    INNER JOIN {catalog}.{schema}.dim_store s ON i.store_id = s.store_id
    LEFT JOIN {catalog}.{schema}.fact_budget b 
        ON i.store_id = b.store_id 
        AND DATE_TRUNC('month', i.invoice_date) = b.budget_month
    LEFT JOIN final_estimates e 
        ON i.order_id = e.order_id
    LEFT JOIN {catalog}.{schema}.dim_estimator est 
        ON e.estimator_id = est.estimator_id
    WHERE i.invoice_date IS NOT NULL
""")

cube_financial_detail.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{schema}.cube_financial_detail")
print(f"✓ Financial Detail Cube created: {catalog}.{schema}.cube_financial_detail")
print(f"Total rows: {cube_financial_detail.count():,}")
fact_count = spark.table(f'{catalog}.{schema}.fact_invoices').filter("invoice_date IS NOT NULL").count()
print(f"Expected rows from fact_invoices: {fact_count:,}")
ratio = cube_financial_detail.count() / fact_count
print(f"Ratio: {ratio:.1f}x {'✓' if ratio == 1.0 else '❌'}")
display(cube_financial_detail.orderBy("invoice_date").limit(10))

In [0]:
cube_survey_detail = spark.sql(f"""
    SELECT 
        sr.store_id,
        s.store_name,
        s.city,
        s.state,
        s.store_type,
        s.manager_name,
        DATE_TRUNC('month', sr.survey_sent_date) AS period_month,
        YEAR(sr.survey_sent_date) AS period_year,
        MONTH(sr.survey_sent_date) AS period_month_num,
        sr.survey_sent_date,
        sr.responded_flag,
        sr.delivered_on_time_rating,
        sr.work_quality_rating,
        sr.cleanliness_rating,
        sr.communication_rating,
        sr.overall_satisfaction_rating
    FROM {catalog}.{schema}.fact_survey_responses sr
    INNER JOIN {catalog}.{schema}.dim_store s ON sr.store_id = s.store_id
    WHERE sr.survey_sent_date IS NOT NULL
""")

cube_survey_detail.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{schema}.cube_survey_detail")
print(f"✓ Survey Detail Cube created: {catalog}.{schema}.cube_survey_detail")
print(f"Total rows: {cube_survey_detail.count():,}")
fact_count = spark.table(f'{catalog}.{schema}.fact_survey_responses').filter("survey_sent_date IS NOT NULL").count()
print(f"Expected rows from fact_survey_responses: {fact_count:,}")
print(f"Ratio: {cube_survey_detail.count() / fact_count:.1f}x")
display(cube_survey_detail.orderBy("survey_sent_date").limit(10))